# Scenario Dreamer Baseline (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/achyutmorang/scenario-dreamer-decision-layer/blob/main/notebooks/scenario_dreamer_baseline_colab.ipynb)

This notebook creates the **baseline experimental substrate** on Colab:
- sync the repo,
- mount Google Drive,
- bind the persistent checkpoint/environment-pack layout,
- install a lean Scenario Dreamer simulation runtime,
- run a smoke-tier baseline evaluation,
- render a demo MP4 while writing run artifacts directly to Drive.


In [1]:
# Step 1: Sync this repo into the Colab runtime
import os
import subprocess
import sys

REPO_URL = 'https://github.com/achyutmorang/scenario-dreamer-decision-layer.git'
REPO_DIR = '/content/scenario-dreamer-decision-layer'

if os.path.isdir(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', 'main'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only', 'origin', 'main'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '-b', 'main', REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
for p in (REPO_DIR, os.path.join(REPO_DIR, 'src')):
    if p not in sys.path:
        sys.path.insert(0, p)

print('repo_rev:', subprocess.check_output(['git', '-C', REPO_DIR, 'rev-parse', '--short', 'HEAD'], text=True).strip())


repo_rev: 4d7e3d8


In [2]:
# Suggested defaults for a Drive-backed smoke baseline
%env SCENARIO_DREAMER_DRIVE_ROOT=/content/drive/MyDrive/scenario_dreamer_research
%env SCENARIO_DREAMER_RUN_SETUP=1
%env SCENARIO_DREAMER_RUN_SMOKE=1
%env SCENARIO_DREAMER_RUN_DEMO=1


env: SCENARIO_DREAMER_DRIVE_ROOT=/content/drive/MyDrive/scenario_dreamer_research
env: SCENARIO_DREAMER_RUN_SETUP=1
env: SCENARIO_DREAMER_RUN_SMOKE=1
env: SCENARIO_DREAMER_RUN_DEMO=1


In [3]:
# Step 2: Mount Drive, clone/pin upstream Scenario Dreamer, and bind Drive-backed assets/results
import json
import os
import subprocess
import sys

from google.colab import drive

drive.mount('/content/drive', force_remount=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', REPO_DIR], check=True)

from scenario_dreamer_decision_layer.colab import bind_drive_layout, inspect_bound_layout

subprocess.run([sys.executable, 'scripts/bootstrap_baseline.py', '--clone-upstream', '--write-lock'], check=True)
binding = bind_drive_layout(os.environ['SCENARIO_DREAMER_DRIVE_ROOT'])
print('binding:')
print(json.dumps(binding, indent=2, sort_keys=True))
print('inspection:')
print(json.dumps(inspect_bound_layout(), indent=2, sort_keys=True))


Mounted at /content/drive
binding:
{
  "bindings": {
    "datasets_root": {
      "actual": "/content/drive/MyDrive/scenario_dreamer_research/scenario_dreamer_decision_layer/assets/datasets",
      "canonical": "/content/scenario-dreamer-decision-layer/artifacts/assets/datasets",
      "mode": "symlink",
      "status": "bound"
    },
    "env_jsons_dir": {
      "actual": "/content/drive/MyDrive/scenario_dreamer_research/scenario_dreamer_decision_layer/assets/scenario_dreamer_waymo_200m_jsons",
      "canonical": "/content/scenario-dreamer-decision-layer/artifacts/assets/scenario_dreamer_waymo_200m_jsons",
      "mode": "symlink",
      "status": "bound"
    },
    "env_pickles_dir": {
      "actual": "/content/drive/MyDrive/scenario_dreamer_research/scenario_dreamer_decision_layer/assets/scenario_dreamer_waymo_200m_pickles",
      "canonical": "/content/scenario-dreamer-decision-layer/artifacts/assets/scenario_dreamer_waymo_200m_pickles",
      "mode": "symlink",
      "status": "bou

In [4]:
# Step 3: Install a lean runtime for Scenario Dreamer simulation on Colab
import os
import subprocess
import sys

RUN_SETUP = os.environ.get('SCENARIO_DREAMER_RUN_SETUP', '1').strip().lower() in {'1', 'true', 'yes', 'y', 'on'}
print('RUN_SETUP:', RUN_SETUP)
if RUN_SETUP:
    subprocess.run([sys.executable, 'scripts/setup_colab_runtime.py', '--editable-project'], check=True)
else:
    print('Skipping runtime setup.')


RUN_SETUP: True


In [5]:
# Step 4: Dry-run the smoke-tier baseline and confirm asset readiness before execution
import json
import subprocess
import sys

raw = subprocess.check_output([sys.executable, 'scripts/run_smoke_eval.py', '--dry-run'], text=True)
print(raw)
smoke_dry_run = json.loads(raw)
missing_assets = smoke_dry_run.get('missing_assets', {})
scenario_count = int(smoke_dry_run.get('scenario_count', 0))
print('smoke_dry_run_summary:')
print(json.dumps({
    'scenario_count': scenario_count,
    'missing_assets': missing_assets,
    'config_snapshot': smoke_dry_run.get('config_snapshot', ''),
}, indent=2, sort_keys=True))
if missing_assets:
    raise RuntimeError(f'Resolve missing assets before Step 5: {missing_assets}')
if scenario_count <= 0:
    raise RuntimeError('Smoke dry-run found zero scenarios. Verify the Scenario Dreamer environment pack binding before Step 5.')



{
  "command": [
    "python",
    "run_simulation.py",
    "sim.mode=scenario_dreamer",
    "sim.dataset_path=/content/drive/MyDrive/scenario_dreamer_research/scenario_dreamer_decision_layer/results/runs/20260408T185831Z_smoke/_subset_pickles",
    "sim.behaviour_model.run_name=ctrl_sim_waymo_1M_steps",
    "sim.behaviour_model.model_path=/content/scenario-dreamer-decision-layer/artifacts/assets/scenario-dreamer/checkpoints/ctrl_sim_waymo_1M_steps/last.ckpt",
    "sim.visualize=True",
    "sim.lightweight=True",
    "sim.verbose=True",
    "sim.simulate_vehicles_only=True",
    "sim.policy=idm",
    "sim.steps=400",
    "sim.dt=0.1",
    "sim.agent_scale=1.0",
    "sim.behaviour_model.tilt=10",
    "sim.behaviour_model.action_temperature=1.0",
    "sim.behaviour_model.use_rtg=True",
    "sim.behaviour_model.predict_rtgs=True",
    "sim.behaviour_model.compute_metrics=False",
    "sim.movie_path=/content/drive/MyDrive/scenario_dreamer_research/scenario_dreamer_decision_layer/results/ru

In [6]:
# Step 5: Execute the smoke-tier baseline run
import json
import os
import subprocess
import sys


def _run_json_cmd(cmd, label):
    proc = subprocess.run(cmd, text=True, capture_output=True, check=False)
    stdout = proc.stdout.strip()
    stderr = proc.stderr.strip()
    if stdout:
        print(stdout)
    if stderr:
        print(f'[{label}_stderr]')
        print(stderr)
    if proc.returncode != 0:
        diag = {
            'exit_code': proc.returncode,
            'failed_command': ' '.join(cmd),
            'results_root': os.environ.get('SCENARIO_DREAMER_RESULTS_ROOT', ''),
            'recent_stdout': stdout[-4000:],
            'recent_stderr': stderr[-4000:],
        }
        print(f'{label}_diagnostics:')
        print(json.dumps(diag, indent=2, sort_keys=True))
        raise subprocess.CalledProcessError(proc.returncode, cmd)
    try:
        return json.loads(stdout)
    except json.JSONDecodeError as exc:
        diag = {
            'failed_command': ' '.join(cmd),
            'results_root': os.environ.get('SCENARIO_DREAMER_RESULTS_ROOT', ''),
            'recent_stdout': stdout[-4000:],
            'recent_stderr': stderr[-4000:],
            'parse_error': str(exc),
        }
        print(f'{label}_diagnostics:')
        print(json.dumps(diag, indent=2, sort_keys=True))
        raise


RUN_SMOKE = os.environ.get('SCENARIO_DREAMER_RUN_SMOKE', '1').strip().lower() in {'1', 'true', 'yes', 'y', 'on'}
print('RUN_SMOKE:', RUN_SMOKE)
smoke_payload = None
if RUN_SMOKE:
    smoke_payload = _run_json_cmd([sys.executable, 'scripts/run_smoke_eval.py'], 'step5_smoke')
    print('smoke_summary:')
    print(json.dumps({
        'run_id': smoke_payload.get('run_id', ''),
        'run_dir': smoke_payload.get('run_dir', ''),
        'scenario_count': smoke_payload.get('scenario_count', 0),
        'metrics': smoke_payload.get('metrics', {}),
    }, indent=2, sort_keys=True))
else:
    print('Skipping smoke run.')



RUN_SMOKE: True
[step5_smoke_stderr]
Traceback (most recent call last):
  File "/content/scenario-dreamer-decision-layer/scripts/run_smoke_eval.py", line 21, in <module>
    raise SystemExit(main())
                     ^^^^^^
  File "/content/scenario-dreamer-decision-layer/scripts/run_smoke_eval.py", line 15, in main
    payload = run_tier("smoke", dry_run=args.dry_run)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/scenario-dreamer-decision-layer/src/scenario_dreamer_decision_layer/runner.py", line 223, in run_tier
    raise RuntimeError(f"Simulation failed with code {proc.returncode}. See {bundle['stderr']}")
RuntimeError: Simulation failed with code 1. See /content/drive/MyDrive/scenario_dreamer_research/scenario_dreamer_decision_layer/results/runs/20260408T185832Z_smoke/stderr.log
step5_smoke_diagnostics:
{
  "exit_code": 1,
  "failed_command": "/usr/bin/python3 scripts/run_smoke_eval.py",
  "recent_stderr": "Traceback (most recent call last):\n  File \"/c

CalledProcessError: Command '['/usr/bin/python3', 'scripts/run_smoke_eval.py']' returned non-zero exit status 1.

In [ ]:
# Step 6: Render a single-environment demo MP4
import json
import os
import subprocess
import sys

RUN_DEMO = os.environ.get('SCENARIO_DREAMER_RUN_DEMO', '1').strip().lower() in {'1', 'true', 'yes', 'y', 'on'}
print('RUN_DEMO:', RUN_DEMO)
demo_payload = None
if RUN_DEMO:
    demo_payload = _run_json_cmd([sys.executable, 'scripts/render_demo.py'], 'step6_demo')
    print('demo_summary:')
    print(json.dumps({
        'run_id': demo_payload.get('run_id', ''),
        'run_dir': demo_payload.get('run_dir', ''),
        'scenario_count': demo_payload.get('scenario_count', 0),
        'metrics': demo_payload.get('metrics', {}),
    }, indent=2, sort_keys=True))
else:
    print('Skipping demo render.')



In [ ]:
# Step 7: Inspect the latest Drive-backed run artifacts and preview the first MP4 if present
import json
import os
from pathlib import Path

from IPython.display import Video, display

results_root = Path(os.environ['SCENARIO_DREAMER_RESULTS_ROOT'])
run_dirs = sorted([p for p in results_root.iterdir() if p.is_dir()], key=lambda p: p.stat().st_mtime, reverse=True)
print('results_root:', results_root)
print('num_run_dirs:', len(run_dirs))
if not run_dirs:
    raise RuntimeError('No run directories found. Execute Step 5 or Step 6 first.')

latest = run_dirs[0]
print('latest_run_dir:', latest)
for name in ['run_manifest.json', 'metrics.json', 'config_snapshot.json', 'video_manifest.json']:
    path = latest / name
    if path.exists():
        print(f'
{name}:')
        print(path.read_text()[:4000])

movies = sorted(latest.rglob('*.mp4'))
print('movie_count:', len(movies))
if movies:
    display(Video(str(movies[0]), embed=True))
